# LLaMAC Dataset Tour for Lab Meeting

**Colab 발표형 데이터 인스펙션 노트북**

- 대상: affective computing / wearable biosignal / ML 모델링 논의 참여자
- 근거: 원 논문 `Scientific Data` 2025 및 Figshare v6 데이터
- 목표: 데이터셋 생성 맥락, 파일 구조, 주요 변수, 예측 target 후보를 빠르게 공유
- 방식: 발표형 요약 + 실제 CSV 구조 확인 + interactive EDA


In [ ]:
# @title 0. 발표형 스타일 배너
from IPython.display import HTML, Markdown, display

HTML("""
<style>
.llamac-hero {
  background: linear-gradient(135deg, #101828 0%, #1d4ed8 48%, #06b6d4 100%);
  color: #fff; padding: 28px 32px; border-radius: 22px; margin: 8px 0 18px 0;
  box-shadow: 0 18px 45px rgba(15,23,42,.22);
  font-family: Inter, Pretendard, -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
}
.llamac-hero h1 {font-size: 34px; margin: 0 0 8px 0; letter-spacing: -0.04em;}
.llamac-hero p {font-size: 16px; opacity: .92; line-height: 1.6; margin: 0;}
.llamac-pill {display:inline-block; background:rgba(255,255,255,.16); border:1px solid rgba(255,255,255,.25);
  padding:6px 10px; border-radius:999px; margin:12px 8px 0 0; font-size:12px;}
.llamac-card-grid {display:grid; grid-template-columns: repeat(auto-fit, minmax(220px, 1fr)); gap:14px; margin: 16px 0;}
.llamac-card {border:1px solid #e5e7eb; border-radius:18px; padding:16px; background:#fff; box-shadow:0 8px 24px rgba(15,23,42,.06);}
.llamac-card h3 {margin:0 0 8px 0; color:#0f172a; font-size:16px;}
.llamac-card p, .llamac-card li {color:#334155; line-height:1.55; font-size:14px;}
.llamac-note {border-left: 5px solid #2563eb; padding: 12px 14px; background:#eff6ff; border-radius:12px; color:#1e3a8a;}
.llamac-warning {border-left: 5px solid #f97316; padding: 12px 14px; background:#fff7ed; border-radius:12px; color:#7c2d12;}
.llamac-small {font-size:12px; color:#64748b;}
</style>
<div class="llamac-hero">
  <h1>LLaMAC 데이터셋: Biosignal → Emotion / Rating Prediction</h1>
  <p>Low-cost biosignal sensors 기반 multimodal affective computing dataset.<br>
  파일 구조, label table, raw waveform, ML target 후보를 Colab에서 직접 확인합니다.</p>
  <span class="llamac-pill">Scientific Data 2025</span>
  <span class="llamac-pill">Figshare DOI 10.6084/m9.figshare.28748696.v6</span>
  <span class="llamac-pill">EEG · GSR · PPG · SKT · RESP</span>
  <span class="llamac-pill">Valence · Arousal · Dominance · Discrete Emotion</span>
</div>
""")


## 1. 원 논문 기반 한 장 요약

- 기준 문헌: `docs/41597_2025_Article_6165.pdf`
- 설명 방식: 핵심 원문 용어 유지 + 한국어 개조식 정리
- 초점: “어떤 실험으로 만든 데이터인가”와 “ML target을 어떻게 잡을 수 있는가”


In [ ]:
# @title 1-a. 논문 요약 카드
HTML("""
<div class="llamac-card-grid">
  <div class="llamac-card">
    <h3>연구 목적</h3>
    <ul>
      <li>audio-visual media 시청 중 emotion response 측정</li>
      <li>biosignal 기반 emotion / liking prediction 자원 구축</li>
      <li>콘텐츠 평가와 affective computing 모델 개발 지원</li>
    </ul>
  </div>
  <div class="llamac-card">
    <h3>데이터셋 차별점</h3>
    <ul>
      <li><b>low-cost / consumer-grade biosignal sensors</b></li>
      <li><b>continuous emotion</b>: Valence, Arousal, Dominance</li>
      <li><b>discrete emotion</b>: neutral, fun, sadness, anger, fear</li>
      <li>100명 이상 규모 + liking/familiarity 설문 포함</li>
    </ul>
  </div>
  <div class="llamac-card">
    <h3>실험 설계</h3>
    <ul>
      <li>114명 참여, 6명 제외</li>
      <li>유효 participant: <b>108명</b></li>
      <li>participant당 50개, 60초 audio-visual stimuli</li>
      <li>trial 직후 self-assessment questionnaire</li>
    </ul>
  </div>
  <div class="llamac-card">
    <h3>센서 구성</h3>
    <ul>
      <li><b>Emotiv Epoc-X</b>: EEG 14ch</li>
      <li><b>Empatica E4</b>: GSR / PPG / SKT</li>
      <li><b>Vernier GDX-RB</b>: RESP</li>
      <li>파일 family: answer / band / eeg / respiration</li>
    </ul>
  </div>
</div>
<div class="llamac-note">
<b>논문 baseline:</b> EEG/GSR/PPG/SKT/RESP feature → LightGBM 5-class emotion classification.<br>
<b>본 노트북 초점:</b> baseline 재현보다 데이터 구조, target 후보, EDA 기반 ML 문제 정의.
</div>
""")


## 2. Colab 환경 설정과 의존성 설치

- repo 내려받기 없음
- Colab 런타임 내부에서 필요한 패키지만 설치
- Figshare DOI에서 데이터 직접 다운로드


In [ ]:
# @title 2. 패키지 설치 / import / 작업 폴더 설정
from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import re
import subprocess
import sys
import time
import urllib.request
import warnings
import zipfile
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from pathlib import Path
from typing import Any

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    ROOT = Path("/content/llamac_labmeeting_demo")
    ROOT.mkdir(parents=True, exist_ok=True)
    os.chdir(ROOT)
    print("Colab 패키지 설치 중입니다. 첫 실행에서는 1~3분 정도 걸릴 수 있습니다.")
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "polars>=1.40.1", "pyarrow>=14", "numpy>=2.0", "pandas>=2.2",
            "itables>=2.2", "plotly>=5.20", "altair>=5.0", "scikit-learn>=1.5", "matplotlib>=3.8",
        ],
        check=True,
    )
else:
    ROOT = Path.cwd().resolve()

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
from IPython.display import HTML, Markdown, display

try:
    import plotly.express as px
    import plotly.graph_objects as go
    PLOTLY_OK = True
except Exception as exc:
    PLOTLY_OK = False
    print("Plotly 미설치: matplotlib fallback을 사용합니다.", exc)

try:
    from itables import init_notebook_mode, show as itable_show
    init_notebook_mode(all_interactive=True)
    ITABLES_OK = True
except Exception as exc:
    ITABLES_OK = False
    print("itables 미설치: 일반 display fallback을 사용합니다.", exc)

try:
    from google.colab import widgets
    COLAB_WIDGETS_OK = True
except Exception:
    widgets = None
    COLAB_WIDGETS_OK = False

RAW_DIR = ROOT / "data" / "raw"
EXTRACTED_DIR = ROOT / "data" / "extracted"
PROCESSED_DIR = ROOT / "data" / "processed"
INDEX_PATH = PROCESSED_DIR / "dataset_index.csv"
SKIP_HEAVY = os.environ.get("LLAMAC_SKIP_HEAVY") == "1"  # 검증용: 다운로드 생략
for path in [RAW_DIR, EXTRACTED_DIR, PROCESSED_DIR]:
    path.mkdir(parents=True, exist_ok=True)

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(20)
print(f"ROOT = {ROOT}")


## 3. Colab Forms: 다운로드 범위 선택

- 빠른 발표: `LIMIT_SUBJECTS=12` 권장
- 전체 구조 확인: `DOWNLOAD_ALL=True`
- 대용량 다운로드 전, 일부 participant로 schema와 시각화 먼저 확인


In [ ]:
# @title 3. 데이터 다운로드 파라미터 { display-mode: "form" }
DOWNLOAD_ALL = False  # @param {type:"boolean"}
LIMIT_SUBJECTS = 12  # @param {type:"integer"}
DOWNLOAD_WORKERS = 4  # @param {type:"integer"}
FORCE_REDOWNLOAD = False  # @param {type:"boolean"}

# DOWNLOAD_ALL=True이면 전체 participant zip을 받습니다. 아니면 LIMIT_SUBJECTS개만 받습니다.
DOWNLOAD_LIMIT = None if DOWNLOAD_ALL else int(LIMIT_SUBJECTS)
print(f"DOWNLOAD_ALL={DOWNLOAD_ALL}, DOWNLOAD_LIMIT={DOWNLOAD_LIMIT}, workers={DOWNLOAD_WORKERS}")


## 4. LLaMAC 데이터셋 다운로드와 압축해제

- Figshare article API 사용
- participant zip 파일 직접 다운로드
- 안전한 압축해제: path traversal 차단
- `dataset_index.csv` 생성: 파일 구조 탐색용 manifest


In [ ]:
# @title 4. Figshare 다운로드 / 압축해제 / dataset_index 생성
ARTICLE_ID = "28748696"
VERSION = "6"
API_URL = f"https://api.figshare.com/v2/articles/{ARTICLE_ID}/versions/{VERSION}"

@dataclass(frozen=True)
class FigshareFile:
    name: str
    size: int
    download_url: str
    md5: str | None = None


def natural_key(text: str | Path) -> list[object]:
    return [int(x) if x.isdigit() else x.lower() for x in re.split(r"(\d+)", str(text))]


def safe_name(name: str) -> str:
    out = name.replace("\\", "/").split("/")[-1]
    if not out or out in {".", ".."}:
        raise ValueError(f"unsafe filename: {name!r}")
    return out


def fetch_figshare_files() -> list[FigshareFile]:
    req = urllib.request.Request(API_URL, headers={"User-Agent": "llamac-colab-labmeeting-demo/1.0"})
    with urllib.request.urlopen(req, timeout=60) as response:
        metadata = json.load(response)
    (RAW_DIR / "llamac_figshare_manifest.json").write_text(json.dumps(metadata, indent=2, ensure_ascii=False))
    files = []
    for item in metadata["files"]:
        files.append(FigshareFile(str(item["name"]), int(item["size"]), str(item["download_url"]), item.get("computed_md5") or item.get("supplied_md5")))
    return files


def md5sum(path: Path) -> str:
    h = hashlib.md5()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def download_one(file: FigshareFile) -> str:
    target = RAW_DIR / safe_name(file.name)
    if target.exists() and not FORCE_REDOWNLOAD and target.stat().st_size == file.size:
        return f"skip {file.name}"
    tmp = target.with_suffix(target.suffix + ".part")
    req = urllib.request.Request(file.download_url, headers={"User-Agent": "llamac-colab-labmeeting-demo/1.0"})
    with urllib.request.urlopen(req, timeout=180) as response, tmp.open("wb") as out:
        while True:
            chunk = response.read(1024 * 1024)
            if not chunk:
                break
            out.write(chunk)
    if tmp.stat().st_size != file.size:
        raise RuntimeError(f"size mismatch for {file.name}")
    if file.md5 and md5sum(tmp) != file.md5:
        raise RuntimeError(f"md5 mismatch for {file.name}")
    tmp.replace(target)
    return f"download {file.name}"


def safe_extract_zip(zip_path: Path, target_dir: Path) -> None:
    marker = target_dir / ".extracted_ok"
    if marker.exists() and not FORCE_REDOWNLOAD:
        return
    target_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        for info in zf.infolist():
            member = Path(info.filename)
            if member.is_absolute() or ".." in member.parts:
                raise ValueError(f"unsafe zip member: {info.filename}")
        zf.extractall(target_dir)
    marker.write_text(time.strftime("%Y-%m-%d %H:%M:%S"))


def build_dataset_index() -> pl.DataFrame:
    rows = []
    for subject_dir in sorted([p for p in EXTRACTED_DIR.iterdir() if p.is_dir()], key=natural_key):
        for path in sorted(subject_dir.rglob("*"), key=natural_key):
            if not path.is_file() or path.name == ".extracted_ok":
                continue
            rows.append({
                "participant_id": subject_dir.name,
                "file_name": path.name,
                "relative_path": str(path.relative_to(EXTRACTED_DIR)),
                "suffix": path.suffix.lower(),
                "size_bytes": path.stat().st_size,
            })
    df = pl.DataFrame(rows) if rows else pl.DataFrame(schema={"participant_id": pl.Utf8, "file_name": pl.Utf8, "relative_path": pl.Utf8, "suffix": pl.Utf8, "size_bytes": pl.Int64})
    df.write_csv(INDEX_PATH)
    return df

if INDEX_PATH.exists() and not FORCE_REDOWNLOAD:
    print(f"이미 준비된 index 사용: {INDEX_PATH}")
    index = pl.read_csv(INDEX_PATH)
elif SKIP_HEAVY:
    print("LLAMAC_SKIP_HEAVY=1 이므로 다운로드를 생략합니다.")
    index = pl.DataFrame(schema={"participant_id": pl.Utf8, "file_name": pl.Utf8, "relative_path": pl.Utf8, "suffix": pl.Utf8, "size_bytes": pl.Int64})
else:
    fig_files = fetch_figshare_files()
    zips = sorted([f for f in fig_files if f.name.lower().endswith(".zip")], key=lambda f: natural_key(f.name))
    selected = zips[:DOWNLOAD_LIMIT] if DOWNLOAD_LIMIT is not None else zips
    print(f"다운로드 zip 수: {len(selected)}")
    with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as pool:
        futures = [pool.submit(download_one, f) for f in selected]
        for i, future in enumerate(as_completed(futures), start=1):
            print(f"[{i}/{len(selected)}] {future.result()}")
    for f in selected:
        zip_path = RAW_DIR / safe_name(f.name)
        safe_extract_zip(zip_path, EXTRACTED_DIR / zip_path.stem)
    index = build_dataset_index()

if index.height:
    display(index.select(pl.len().alias("files"), pl.col("participant_id").n_unique().alias("participants"), (pl.col("size_bytes").sum() / 1024**2).round(2).alias("size_mib")))
else:
    print("index가 비어 있습니다. Colab에서 다운로드 셀을 실행하면 실제 구조가 표시됩니다.")


## 5. 표와 label helper

- `itables`: 검색/정렬 가능한 interactive table
- `pandas.Styler`: 색상 있는 fallback table
- label helper: `NoVideo → IntendedType`, `EmotType → ReportedType`


In [ ]:
# @title 5. 표/라벨 표시 helper
EMOTION_ID_TO_LABEL = {1: "neutral", 2: "fun", 3: "sadness", 4: "anger", 5: "fear"}
EMOTION_LABELS = [EMOTION_ID_TO_LABEL[i] for i in range(1, 6)]
RATING_COLS = ["Valence", "Arousal", "Dominance", "Liking", "EmotStr", "Seen"]


def show_table(data: pl.DataFrame | pd.DataFrame, caption: str = "", max_rows: int = 30) -> None:
    pdf = data.to_pandas() if isinstance(data, pl.DataFrame) else data
    if caption:
        display(HTML(f"<div class='llamac-note'><b>{caption}</b></div>"))
    if ITABLES_OK and len(pdf) <= 5000:
        itable_show(pdf, classes="display compact", maxBytes=0)
    else:
        display(pdf.head(max_rows).style.background_gradient(cmap="Blues", axis=None))


def file_family(name: str) -> str:
    if name == "answer.csv":
        return "answer"
    if name.startswith("band_"):
        return "band: GSR/PPG/SKT"
    if name.startswith("eeg_"):
        return "eeg: 14ch EEG"
    if name.startswith("respiration_"):
        return "respiration: belt force"
    return "other"


def intended_from_novideo(no_video: int | float | None) -> int | None:
    if no_video is None:
        return None
    no_video = int(no_video)
    if 1 <= no_video <= 10:
        return 1
    if 11 <= no_video <= 20:
        return 2
    if 21 <= no_video <= 30:
        return 3
    if 31 <= no_video <= 40:
        return 4
    if 41 <= no_video <= 50:
        return 5
    return None


def add_labels(frame: pl.DataFrame) -> pl.DataFrame:
    return frame.with_columns(
        pl.col("NoVideo").map_elements(intended_from_novideo, return_dtype=pl.Int64).alias("IntendedType"),
        pl.col("EmotType").cast(pl.Int64, strict=False).alias("ReportedType"),
    ).with_columns(
        pl.col("IntendedType").replace_strict(EMOTION_ID_TO_LABEL, default="unknown").alias("IntendedLabel"),
        pl.col("ReportedType").replace_strict(EMOTION_ID_TO_LABEL, default="unknown").alias("ReportedLabel"),
    )


## 6. 데이터 파일 구조 인스펙션

- 단위: participant folder
- 구성: `answer.csv` + trial별 biosignal CSV
- 목적: 모델링 전 데이터 granularity와 파일 family 확인


In [ ]:
# @title 6. 파일 family / participant 구조 보기
if index.height == 0:
    print("index가 비어 있어 파일 구조를 표시할 수 없습니다.")
else:
    index2 = index.with_columns(pl.col("file_name").map_elements(file_family, return_dtype=pl.Utf8).alias("family"))
    family_summary = (
        index2.group_by("family")
        .agg(
            pl.len().alias("file_count"),
            pl.col("participant_id").n_unique().alias("participants"),
            (pl.col("size_bytes").sum() / 1024**2).round(2).alias("size_mib"),
            pl.col("file_name").n_unique().alias("unique_names"),
        )
        .sort("family")
    )
    participant_summary = (
        index2.group_by(["participant_id", "family"])
        .agg(pl.len().alias("files"))
        .pivot(index="participant_id", on="family", values="files", aggregate_function="first")
        .fill_null(0)
        .sort("participant_id")
    )

    if COLAB_WIDGETS_OK:
        tab = widgets.TabBar(["파일 family 요약", "participant별 파일 수", "원본 index 샘플"])
        with tab.output_to("파일 family 요약"):
            show_table(family_summary, "파일 종류별 개수와 용량")
        with tab.output_to("participant별 파일 수"):
            show_table(participant_summary, "participant별 answer/band/eeg/respiration 파일 수")
        with tab.output_to("원본 index 샘플"):
            show_table(index2.head(50), "dataset_index.csv 샘플")
    else:
        show_table(family_summary, "파일 종류별 개수와 용량")
        show_table(participant_summary.head(20), "participant별 파일 수: 앞 20명")
        show_table(index2.head(30), "dataset_index.csv 샘플")


## 7. `answer.csv`: ML target 후보 table

- trial-level questionnaire table
- continuous emotion rating: Valence / Arousal / Dominance
- discrete emotion response: ReportedType
- stimulus-derived label: IntendedType


In [ ]:
# @title 7. answer.csv 통합과 label table 보기
answers = pl.DataFrame()
if index.height == 0:
    print("index가 비어 있어 answer table을 만들 수 없습니다.")
else:
    tables = []
    for row in index.filter(pl.col("file_name") == "answer.csv").sort("participant_id").iter_rows(named=True):
        path = EXTRACTED_DIR / row["relative_path"]
        tables.append(pl.read_csv(path).with_columns(pl.lit(int(row["participant_id"])).alias("SubjectID")))
    answers = add_labels(pl.concat(tables, how="diagonal_relaxed")) if tables else pl.DataFrame()
    label_view = answers.select([
        "SubjectID", "Trial", "NoVideo", "IntendedType", "IntendedLabel", "ReportedType", "ReportedLabel",
        "Valence", "Arousal", "Dominance", "Liking", "EmotStr", "Seen",
    ])
    show_table(label_view.head(80), "Intended label과 reported label이 함께 있는 trial-level table")

    HTML("""
    <div class="llamac-card-grid">
      <div class="llamac-card"><h3>Continuous / dimensional</h3><p><b>Valence</b>: pleasant ↔ unpleasant, <b>Arousal</b>: calm ↔ activated, <b>Dominance</b>: submissive ↔ dominant. 모두 1~5 rating입니다.</p></div>
      <div class="llamac-card"><h3>Discrete / categorical</h3><p><b>ReportedType</b>은 참가자의 자기보고 감정입니다. <b>IntendedType</b>은 NoVideo 자극 설계에서 온 label입니다.</p></div>
      <div class="llamac-card"><h3>논의 포인트</h3><p>ML target은 ReportedType, Valence/Arousal/Dominance, Liking 모두 가능하지만, split은 participant-grouped로 해야 subject leakage를 피할 수 있습니다.</p></div>
    </div>
    """)


## 8. `band_*.csv`: GSR / PPG / SKT 구조와 raw waveform

- wearable peripheral signal file
- 컬럼: GSR, PPG, SKT + 각 timestamp
- 확인: dataframe head + raw waveform sanity plot


In [ ]:
# @title 8. band sensor dataframe과 raw waveform sanity plot
band_df = pl.DataFrame()
if index.height == 0:
    print("index가 비어 있어 band 파일을 표시할 수 없습니다.")
else:
    band_row = index.filter(pl.col("file_name").str.starts_with("band_")).sort(["participant_id", "file_name"]).row(0, named=True)
    band_path = EXTRACTED_DIR / band_row["relative_path"]
    band_df = pl.read_csv(band_path)
    print(f"sample band file = participant {band_row['participant_id']} / {band_row['file_name']}")
    print(f"shape = {band_df.shape}")
    print(f"columns = {band_df.columns}")
    show_table(band_df.head(40), "band_*.csv dataframe head: GSR / PPG / SKT와 각 timestamp")

    wave = band_df.select(["GSR", "GSR_Time", "PPG", "PPG_Time", "SKT", "SKT_Time"]).to_pandas()
    if PLOTLY_OK:
        fig = go.Figure()
        for value_col, time_col in [("GSR", "GSR_Time"), ("PPG", "PPG_Time"), ("SKT", "SKT_Time")]:
            t = wave[time_col] - wave[time_col].iloc[0]
            fig.add_trace(go.Scatter(x=t, y=wave[value_col], mode="lines", name=value_col))
        fig.update_layout(title="Raw waveform sanity plot: GSR / PPG / SKT", xaxis_title="seconds from trial start", yaxis_title="raw sensor value", height=430)
        fig.show()
    else:
        fig, axes = plt.subplots(3, 1, figsize=(10, 6))
        for ax, value_col, time_col in zip(axes, ["GSR", "PPG", "SKT"], ["GSR_Time", "PPG_Time", "SKT_Time"], strict=True):
            ax.plot(wave[time_col] - wave[time_col].iloc[0], wave[value_col])
            ax.set_title(value_col)
        fig.tight_layout(); plt.show()


## 9. EEG / respiration 파일 구조 확인

- EEG: Timestamp + 14-channel EEG
- RESP: Time + belt force signal
- 본 노트북 역할: schema와 head 중심 확인


In [ ]:
# @title 9. eeg_*.csv / respiration_*.csv 구조 보기
if index.height == 0:
    print("index가 비어 있어 EEG/RESP 파일을 표시할 수 없습니다.")
else:
    eeg_row = index.filter(pl.col("file_name").str.starts_with("eeg_")).sort(["participant_id", "file_name"]).row(0, named=True)
    resp_row = index.filter(pl.col("file_name").str.starts_with("respiration_")).sort(["participant_id", "file_name"]).row(0, named=True)
    eeg_df = pl.read_csv(EXTRACTED_DIR / eeg_row["relative_path"], n_rows=8)
    resp_df = pl.read_csv(EXTRACTED_DIR / resp_row["relative_path"], n_rows=8)
    show_table(eeg_df, "eeg_*.csv head: Timestamp + 14 EEG channels")
    show_table(resp_df, "respiration_*.csv head: Time + Force signal")


## 10. Intended label × Reported label

- IntendedType: 자극 설계상 의도된 감정
- ReportedType: 참가자가 실제 보고한 감정
- 핵심 질문: 모델 target을 stimulus label로 둘 것인가, self-report label로 둘 것인가


In [ ]:
# @title 10. intended-vs-reported heatmap
if answers.height == 0:
    print("answers가 비어 있어 heatmap을 그릴 수 없습니다.")
else:
    counts = answers.group_by(["IntendedType", "ReportedType"]).agg(pl.len().alias("count"))
    matrix = np.zeros((5, 5), dtype=float)
    for intended, reported, count in counts.iter_rows():
        if intended in EMOTION_ID_TO_LABEL and reported in EMOTION_ID_TO_LABEL:
            matrix[int(intended) - 1, int(reported) - 1] = count
    if PLOTLY_OK:
        fig = px.imshow(matrix, x=EMOTION_LABELS, y=EMOTION_LABELS, text_auto=True, color_continuous_scale="YlGnBu", aspect="auto")
        fig.update_layout(title="Intended emotion × Reported emotion", xaxis_title="reported emotion", yaxis_title="intended emotion", height=470)
        fig.show()
    else:
        fig, ax = plt.subplots(figsize=(6, 5))
        im = ax.imshow(matrix, cmap="YlGnBu"); fig.colorbar(im, ax=ax)
        ax.set_xticks(range(5), EMOTION_LABELS, rotation=45, ha="right"); ax.set_yticks(range(5), EMOTION_LABELS)
        ax.set_title("Intended emotion × Reported emotion")
        fig.tight_layout(); plt.show()


## 11. Valence / Arousal / Dominance와 label 상관관계

- VAD는 예측 target으로 설정 가능
- 해석: 성격 특성보다는 60초 자극 직후의 trial-level self-assessed affect rating
- ML 형태: regression 또는 1~5 ordinal classification
- 주의: 개인차, 순서형 척도, PPG-only 신호대잡음비


In [ ]:
# @title 11. rating/label correlation heatmap + Valence-Arousal scatter
if answers.height == 0:
    print("answers가 비어 있어 상관관계를 표시할 수 없습니다.")
else:
    corr_cols = ["IntendedType", "ReportedType", "Valence", "Arousal", "Dominance", "Liking", "EmotStr", "Seen"]
    corr = answers.select(corr_cols).corr()
    corr_pdf = corr.to_pandas()
    corr_pdf.index = corr_cols
    corr_pdf.columns = corr_cols
    if PLOTLY_OK:
        fig = px.imshow(corr_pdf, text_auto=".2f", color_continuous_scale="RdBu_r", zmin=-1, zmax=1, aspect="auto")
        fig.update_layout(title="Rating / label Pearson correlation", height=560)
        fig.show()
        scatter_df = answers.select(["Valence", "Arousal", "Dominance", "ReportedLabel", "IntendedLabel", "SubjectID", "Trial"]).to_pandas()
        fig2 = px.scatter(scatter_df, x="Valence", y="Arousal", color="ReportedLabel", symbol="IntendedLabel", hover_data=["SubjectID", "Trial", "Dominance"], title="Valence-Arousal plane: reported label 색상, intended label 모양")
        fig2.update_layout(height=520)
        fig2.show()
    else:
        display(corr_pdf.style.background_gradient(cmap="RdBu_r", axis=None, vmin=-1, vmax=1).format("{:.2f}"))


## 12. ML 문제 정의 후보

- 목적: 예측 target과 baseline model 후보 정리
- 논의 축: classification vs regression vs ordinal prediction
- 평가 원칙: participant-grouped split로 subject leakage 방지


In [ ]:
# @title 12. Target / model 후보 테이블
ml_plan = pd.DataFrame([
    {
        "target": "ReportedType",
        "type": "5-class classification",
        "meaning": "참가자가 실제 보고한 discrete emotion",
        "baseline_models": "Logistic Regression, RandomForest/ExtraTrees, LightGBM, XGBoost",
        "advanced_models": "ROCKET/MiniROCKET, 1D-CNN, TCN, InceptionTime",
        "caution": "participant-grouped split 필수; intended label과 혼동 금지",
    },
    {
        "target": "Valence / Arousal / Dominance",
        "type": "regression 또는 ordinal classification",
        "meaning": "각 trial 직후의 dimensional affect rating",
        "baseline_models": "Ridge/ElasticNet, RandomForestRegressor, LightGBMRegressor",
        "advanced_models": "multi-task regression, ordinal models, sequence models",
        "caution": "성격 특성이라기보다 순간 self-assessment; 개인차 보정 필요",
    },
    {
        "target": "Liking",
        "type": "3-class 또는 ordinal classification",
        "meaning": "콘텐츠를 더 보고 싶은지에 가까운 선호도",
        "baseline_models": "Ordinal/logistic regression, tree ensembles",
        "advanced_models": "emotion + biosignal multi-task learning",
        "caution": "emotion보다 콘텐츠 취향/친숙도 confounding 가능",
    },
    {
        "target": "Seen",
        "type": "binary classification / confounder",
        "meaning": "자극을 이미 본 적 있는지 familiarity",
        "baseline_models": "confounder analysis, stratified evaluation",
        "advanced_models": "domain adaptation / covariate adjustment",
        "caution": "주 target보다는 confounder로 다루는 편이 자연스러움",
    },
])

if ITABLES_OK:
    itable_show(ml_plan, classes="display compact", maxBytes=0)
else:
    display(ml_plan.style.set_caption("ML target 후보").background_gradient(cmap="Blues", subset=["target"]))

HTML("""
<div class="llamac-warning">
<b>논의 포인트:</b> 스마트워치/PPG-only 적용을 전제하면 ReportedType 5-class만 고집하지 않기.<br>
- Valence/Arousal coarse binary 또는 ordinal target 검토<br>
- participant-normalization 적용 여부 비교<br>
- grouped CV 기준으로 일반화 성능 확인
</div>
""")


## 13. 발표 마무리 체크리스트

- 데이터셋: intended emotion과 reported emotion을 모두 포함
- VAD: 예측 가능하나 trial-level self-report로 해석
- Wearable 적용: PPG-only 또는 PPG+SKT부터 시작하는 전략이 현실적
- 모델링 순서: tabular baseline → time-series baseline → 개인화/정규화 전략
